<a href="https://colab.research.google.com/github/jiffahnisrina/Preprocessing-Movie-Poster/blob/main/Preprocessing_Movie_Poster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text and Image Preprocessing**

**Text Preprocessing**

TBP Manajemen Big Data
Kelas A
Anggota Kelompok 4 :
1. Aneira Nasywa Azalia  (M0724021)
2. Aogasti Lativa Yaquti (M0724049)
3. Jihan Iffah Nisrina   (M0724061)
4. Linka Indah Cantika	 (M0724063)

**Dataset Movie Poster**

- **Deskripsi :**
 > Dataset Movie Posters-100k berisi kumpulan data film dengan berbagai kolom informasi yang memuat detail lengkap tentang film-film tersebut. Data ini terdiri dari entri film yang mencakup informasi identitas, visual, deskriptif, finansial, dan teknis. Struktur datanya menunjukkan karakteristik umum data film yang komprehensif, di mana setiap baris merepresentasikan satu film dengan atribut-atribut yang beragam.
- **Sumber Data :**
 > Hugging Face
https://huggingface.co/datasets/skvarre/movie_posters-100k

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

In [ ]:
import pandas as pd

# Load parquet
df = pd.read_parquet("/content/drive/MyDrive/MBD/movie_poster.parquet")

# Simpan ke CSV
df.to_csv("/content/drive/MyDrive/MBD/movie_poster.csv", index=False)

Membaca data csv

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/MBD/movie_poster.csv")
df.head()

In [ ]:
df.shape

Cek apakah ada data yang hilang

In [ ]:
df.isnull().sum()

Hapus data yang hilang


In [ ]:
df = df.dropna()

cek apakah ada data duplikat

In [ ]:
df.duplicated().sum()

Setelah dataset dibersihkan yang awalnya ada 1083 baris/instances menjadi 819 baris/instances.

In [ ]:
df.shape

Menampilkan kolom text secara penuh dan menampilkan 5 baris data pertama

In [ ]:
from IPython.display import display

display(df.head())

Kali ini kami akan menggunakan kolom "overview" untuk text preprocessing.

# **1. Remove Punctuation**

Sebelum

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df[["overview"]].head()

Sesudah

In [ ]:
import string

text_rp = df["overview"].str.translate(
    str.maketrans('', '', string.punctuation)
)

text_rp.head(5)

In [ ]:
import string

extra_punct = "—–…“”‘’"
all_punct = string.punctuation + extra_punct

text_rp1 = df["overview"].str.translate(
    str.maketrans("", "", all_punct)
)

text_rp1.head(5)

Proses ini digunakan untuk menghapus semua tanda baca karena tidak memberikan informasi penting dan hanya menambah noise pada teks. Pada proses ini menggunakan str.translate() dan str.maketrans() yang berfungsi menghapus seluruh karakter pada daftar string.punctuation.

Selain itu, ditambahkan pula beberapa tanda baca non-standar (—, –, …, “ ”, ‘ ’) ke dalam variabel extra_punct, sehingga proses pembersihan menjadi lebih lengkap dan mampu menghilangkan seluruh simbol yang tidak relevan dari kolom overview.

# **2. Remove Stopwords**

In [ ]:
# setting up NLTK
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

text_rs = text_rp1.apply(lambda x: ' '.join(
    [word for word in word_tokenize(x) if word.lower() not in stop_words]
))

text_rs.head(5)

Proses ini digunakan untuk menghapus kata-kata umum yang frekuensinya tinggi tetapi tidak memiliki kontribusi penting terhadap makna, sehingga teks menjadi lebih ringkas dan informatif. Pada proses ini memanfaatkan daftar stopwords dari NLTK melalui stopwords.words('english'), kemudian setiap teks di-tokenize dengan word_tokenize(). Proses penyaringan dilakukan dengan mempertahankan hanya kata yang tidak termasuk dalam daftar stopwords, dan hasilnya digabungkan kembali menjadi kalimat melalui fungsi join().

Karena dataset yang digunakan berupa text overview, maka isinya umumnya bersifat formal dan tidak mengandung kata-kata slang, sehingga proses penghapusan stopwords berjalan lebih efektif dan tidak berisiko menghilangkan istilah khusus atau bahasa gaul yang mungkin relevan.

# **3. Setting into lower case**

In [ ]:
text_to_lowercase = text_rs

new_text_cleaned = text_to_lowercase.str.lower()
from IPython.display import display

display(new_text_cleaned.to_frame().head())

Langkah ini bertujuan untuk mengubah semua huruf kapital dalam teks menjadi huruf kecil. Dalam notebook ini, ini dilakukan pada new_text_cleaned = text_to_lowercase.str.lower(), yang beroperasi pada teks setelah penghapusan tanda baca dan stopword.

# **4. Stemming**

In [ ]:
# Stemming
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

ps = PorterStemmer()
text_for_stemming = new_text_cleaned

# Mengaplikasikan fungsi stemming kustom untuk setiap entri teks
text_after_stemming = text_for_stemming.apply(lambda x: ' '.join([ps.stem(w) for w in word_tokenize(x)]))
from IPython.display import display

display(text_after_stemming.to_frame().head())

Stemming adalah proses untuk mengurangi kata-kata ke bentuk akar atau dasarnya. Proses ini sering menghilangkan akhiran dan awalan, yang dapat menghasilkan kata-kata yang bukan kata bahasa Inggris yang sebenarnya atau yang terdengar aneh. Tujuan dari stemming adalah untuk menormalisasi kata-kata sehingga variasi dari suatu kata diperlakukan sebagai kata yang sama, yang berguna untuk tugas-tugas seperti analisis teks dan pengambilan informasi.

Sebagai contoh:

discover menjadi discov
violent menjadi violent (tidak ada perubahan di sini, tetapi kata lain mungkin berubah)
story menjadi stori
Ini adalah karakteristik umum dari algoritma stemming seperti Porter Stemmer, yang berfokus pada variasi morfologi daripada menghasilkan kata-kata yang sempurna secara linguistik. Jika Anda membutuhkan bentuk akar yang lebih akurat, Anda dapat mempertimbangkan lemmatisasi, yang menggunakan kosakata dan analisis morfologi kata untuk mengembalikan bentuk dasar atau kamus dari suatu kata, yang dikenal sebagai lemma.

# **5. Lemmatization**

In [ ]:
# Lemmatization
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

wnl = WordNetLemmatizer()

In [ ]:
# Gunakan hasil dari tahap sebelumnya (setelah lowercase)
lemmatized_text = []

for review in new_text_cleaned:
    words = word_tokenize(review)
    lemmas = [wnl.lemmatize(w) for w in words]
    lemmatized_text.append(' '.join(lemmas))

In [ ]:
# Tampilkan 5 baris pertama
print("Setelah lemmatization (5 baris pertama):\n")
for r in lemmatized_text[:5]:
    print(r)

Tahap ini dilakukan untuk mengubah setiap kata ke bentuk dasarnya tanpa mengubah makna kalimat. Proses lemmatization membantu menyatukan kata-kata yang sebenarnya memiliki arti sama, seperti falls menjadi fall atau controls menjadi control, sehingga tidak dihitung sebagai kata yang berbeda. Pada hasil lemmatization, beberapa kata dalam teks seperti falls, turns, atau comes telah direduksi menjadi bentuk dasar (fall, turn, come), sementara kata-kata khusus seperti nama tokoh, lokasi, dan istilah unik tetap dipertahankan agar konteks cerita tidak hilang.

# **6. Tokenization**

In [ ]:
from nltk.tokenize import word_tokenize
text_for_tokenizing = lemmatized_text
tokenized_reviews = [word_tokenize(review) for review in text_for_tokenizing]

# Hitung total token
total_tokens = sum(len(tokens) for tokens in tokenized_reviews)

print("Jumlah token:", total_tokens)
print("\nHasil tokenization (5 baris pertama):\n")

# Tampilkan hanya 5 baris pertama
for i, tokens in enumerate(tokenized_reviews[:5]):
    print(f"Review {i+1}:")
    print(tokens)
    print("-" * 80)


Tahap ini memecah setiap review menjadi token, yaitu potongan kata-kata yang berdiri sendiri. Tokenisasi dilakukan per baris karena setiap baris merepresentasikan satu review film yang berbeda, sehingga proses analisis seperti penghitungan frekuensi kata, pencarian kata penting, ataupun identifikasi tema dapat dilakukan secara lebih terstruktur. Pada hasil tokenisasi, setiap review diubah menjadi daftar kata seperti ethan, hunt, dangerous, atau mission, sehingga memudahkan proses analisis lanjutan yang membutuhkan representasi teks dalam bentuk elemen-elemen kata yang terpisah.

# Tampilan text awal dan setelah di tokenize

In [ ]:
import pandas as pd

# Simpan pengaturan max_colwidth saat ini
original_max_colwidth = pd.get_option('display.max_colwidth')

# Atur max_colwidth untuk menampilkan sebagian saja
pd.set_option('display.max_colwidth', 100) # Anda bisa menyesuaikan angka ini

# Add the tokenized_reviews as a new column to df
df['tokenized_reviews_col'] = tokenized_reviews

# Tampilkan 5 baris pertama dari kolom 'overview' dan 'tokenized_reviews_col'
display(df[['overview', 'tokenized_reviews_col']].head())

# Kembalikan pengaturan max_colwidth ke nilai semula (atau None jika ingin full lagi)
pd.set_option('display.max_colwidth', original_max_colwidth)

# **7. Vectorization**

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

joined_reviews = [" ".join(tokens) for tokens in tokenized_reviews]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(joined_reviews)

print("Vocabulary:", vectorizer.get_feature_names_out())
print("TF-IDF Matrix:")
print(X.toarray())

Dalam bentuk dataframe



In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Gabungkan token menjadi kalimat
joined_reviews = [" ".join(tokens) for tokens in tokenized_reviews]

# TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(joined_reviews)

# Ambil nama fitur (kata)
feature_names = vectorizer.get_feature_names_out()

# Konversi ke DataFrame
tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=feature_names
)

# Tampilkan 5 baris pertama
display(tfidf_df.head())


Pada tahap vectorization, setiap review yang telah melalui proses tokenisasi diubah menjadi representasi numerik menggunakan metode TF-IDF (Term Frequency–Inverse Document Frequency). Proses ini dilakukan per baris karena setiap baris merepresentasikan satu review film yang berbeda, sehingga setiap review akan dipetakan ke dalam satu vektor numerik. Nilai TF-IDF mencerminkan tingkat kepentingan suatu kata dalam sebuah review, di mana kata yang sering muncul pada satu review tetapi jarang muncul pada review lain akan memiliki bobot yang lebih tinggi. Dengan demikian, hasil vectorization berupa matriks angka yang mampu merepresentasikan karakteristik setiap review secara kuantitatif, sehingga data teks dapat digunakan dalam analisis lanjutan seperti pemodelan machine learning, pengukuran kemiripan dokumen, maupun klasifikasi teks.

**Image Preprocessing**

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/MBD/movie_poster.csv")
df.head()

In [ ]:
df.shape

Cleaning Datase

In [ ]:
# Cek missing values
df.isnull().sum()

In [ ]:
# hapus missing value
df = df.dropna()

In [ ]:
# cek duplikat
df.duplicated().sum()

Setelah dataset dibersihkan yang awalnya ada 1083 baris/instances menjadi 819 baris/instances.

In [ ]:
# setelah dataset dibersihkan
df.shape

Kami akan menggunakan kolom "image" untuk image preprocessing.

Sebelum menampilkan perbandingan, kita perlu membuat dataset_cv_initial yang berisi gambar-gambar asli dalam format OpenCV (numpy array) agar konsisten.

In [ ]:
import cv2
import numpy as np
import ast

dataset_cv_initial = []
print("Memuat gambar asli ke dataset_cv_initial...")
for index, row in df.iterrows():
    # Convert the string representation of the dictionary to an actual dictionary
    image_data = ast.literal_eval(row["image"])
    img_bytes = image_data["bytes"]
    np_array = np.frombuffer(img_bytes, np.uint8)
    cv2_img = cv2.imdecode(np_array, cv2.IMREAD_UNCHANGED)
    dataset_cv_initial.append(cv2_img)

print(f"Selesai memuat {len(dataset_cv_initial)} gambar asli.")

# **1. Resize Image**

Untuk poster film, yang umumnya merupakan gambar detail dengan teks, wajah, dan karya seni, interpolasi Cubic (cv2.INTER_CUBIC) umumnya merupakan pilihan yang paling cocok.

In [ ]:
import cv2

dataset_cv_step1 = []

for original_img_cv in dataset_cv_initial:
    resized_img_cv = cv2.resize(
        original_img_cv,
        (224, 224),
        interpolation=cv2.INTER_CUBIC
    )
    dataset_cv_step1.append(resized_img_cv)

In [ ]:
import matplotlib.pyplot as plt

print("Perbandingan 2 gambar pertama (Original vs Resized - Cubic):")

for i in range(2):
    original_img_cv = dataset_cv_initial[i]
    resized_img_cv  = dataset_cv_step1[i]

    original_img_rgb = cv2.cvtColor(original_img_cv, cv2.COLOR_BGR2RGB)
    resized_img_rgb  = cv2.cvtColor(resized_img_cv, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(original_img_rgb)
    plt.title(f"Gambar Asli ke-{i+1}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(resized_img_rgb)
    plt.title(f"1. Resized ke-{i+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# **2. Brightness Adjustment**

Penggunaan brightness adjustment dengan metode gain (perkalian) dipilih karena dapat mempertegas perbedaan tingkat kecerahan antar piksel, sehingga karakter visual poster film yang kuat dan dramatis dapat dipertahankan.

In [ ]:
dataset_cv_step2 = []

for resized_img in dataset_cv_step1:
    gain_img = cv2.convertScaleAbs(resized_img, alpha=1.5, beta=0)
    dataset_cv_step2.append(gain_img)

In [ ]:
import matplotlib.pyplot as plt
import cv2

print("Perbandingan 2 gambar pertama (Resized vs Gain Adjusted):")

for i in range(2):
    resized_img = dataset_cv_step1[i]
    gain_img = dataset_cv_step2[i]

    resized_rgb = cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB)
    gain_rgb = cv2.cvtColor(gain_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(resized_rgb)
    plt.title(f"1. Resized ke-{i+1}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(gain_rgb)
    plt.title(f"2. Gain Adjusted ke-{i+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# **3. Color normalization / Gamma Correction**

Pada tahap ini digunakan gamma correction untuk menyeimbangkan pencahayaan pada gambar. Teknik ini membantu mencerahkan bagian yang terlalu gelap dan menahan bagian yang terlalu terang, sehingga detail dalam poster tetap terlihat jelas. Dengan penyesuaian ini, gambar menjadi lebih natural.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# GAMMA CORRECTION
gamma = 1.5
inv_gamma = 1.0 / gamma

table = np.array([((i / 255.0) ** inv_gamma) * 255
                  for i in range(256)]).astype("uint8")

dataset_cv_step3 = []

for img in dataset_cv_step2:
    gamma_img = cv2.LUT(img, table)
    dataset_cv_step3.append(gamma_img)

print("Perbandingan 2 gambar pertama (Gain Adjusted vs Gamma Correction):")

for i in range(2):
    bright_img = dataset_cv_step2[i]
    gamma_img  = dataset_cv_step3[i]

    plt.figure(figsize=(12, 6))

    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(bright_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Brightness ke-{i+1}")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(cv2.cvtColor(gamma_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Gamma Corrected ke-{i+1}")
    plt.axis("off")

    plt.show()

# **4. Contrast Enhancement (CLAHE)**

Tahap selanjutnya adalah peningkatan kontras menggunakan CLAHE. Metode ini bekerja secara adaptif, setiap area kecil pada gambar ditingkatkan kontrasnya sesuai kebutuhan. Hasilnya, elemen penting seperti teks, wajah, dan objek visual pada poster terlihat lebih tegas tanpa membuat gambar menjadi berlebihan. Proses ini membantu sistem mengenali fitur secara lebih akurat.

In [ ]:
import cv2
import matplotlib.pyplot as plt

dataset_cv_step4 = []

# CLAHE
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

for img in dataset_cv_step3:
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)
    L_clahe = clahe.apply(L)
    lab_clahe = cv2.merge((L_clahe, A, B))
    clahe_img = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

    dataset_cv_step4.append(clahe_img)

In [ ]:
print("Perbandingan 2 gambar pertama (Gamma Correction vs Contrast Enhancenment):")

for i in range(2):
    gamma_img = dataset_cv_step3[i]
    clahe_img = dataset_cv_step4[i]

    gamma_rgb = cv2.cvtColor(gamma_img, cv2.COLOR_BGR2RGB)
    clahe_rgb = cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(gamma_rgb)
    plt.title(f"3. Gamma Corrected {i+1}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(clahe_rgb)
    plt.title(f"4. CLAHE Enhanced {i+1}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# **5. Noise Reduction**

Tahap berikutnya adalah proses pengurangan noise ringan (Light Noise Reduction) menggunakan Bilateral Filter. Metode ini dipilih karena mampu mengurangi bintik-bintik noise tanpa mengaburkan detail penting pada gambar. Tidak seperti metode blur biasa yang dapat merusak ketajaman teks dan garis, bilateral filter mempertahankan tepi (edges) sehingga elemen visual pada poster seperti tulisan, kontur wajah, dan objek utama tetap terlihat jelas.

In [ ]:
import cv2
import matplotlib.pyplot as plt

dataset_cv_step5 = []
for img in dataset_cv_step4:
    # bilateralFilter(src, diameter, sigmaColor, sigmaSpace)
    denoised_img = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)
    dataset_cv_step5.append(denoised_img)

print("Perbandingan 2 gambar pertama (CLAHE vs Light Noise Reduction):\n")

for i in range(2):
    clahe_img = dataset_cv_step4[i]
    denoised_img = dataset_cv_step5[i]

    clahe_rgb = cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)
    denoised_rgb = cv2.cvtColor(denoised_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clahe_rgb)
    plt.title(f"4. CLAHE Enhanced {i+1}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(denoised_rgb)
    plt.title(f"5. Light Noise Reduced {i+1}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# **6. Normalization** (channel-wise normalization)


Terapkan normalisasi pada gambar di dataset_cv_step5 untuk menskalakan intensitas piksel ke rentang standar.


In [ ]:
import numpy as np
import cv2

# Define mean and std for channel-wise normalization (common for ImageNet, often used as a baseline)
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

dataset_cv_step6 = []

for img in dataset_cv_step5:
    # Convert BGR to RGB as mean/std are typically defined for RGB channels
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert to float32 and scale to [0, 1]
    img_float = img_rgb.astype(np.float32) / 255.0

    # Apply channel-wise normalization: (image - mean) / std
    # Reshape mean and std for broadcasting across image dimensions (H, W, C)
    normalized_img = (img_float - mean) / std
    dataset_cv_step6.append(normalized_img)

print(f"Channel-wise normalized {len(dataset_cv_step6)} images and stored in dataset_cv_step6.")

Langkah selanjutnya adalah menampilkan perbandingan visual dari 5 gambar sebelum dan sesudah normalisasi untuk mengamati pengaruhnya terhadap distribusi piksel gambar. Menggunakan matplotlib.pyplot untuk memplot gambar dari dataset_cv_step5 (sebelum normalisasi) dan dataset_cv_step6 (setelah normalisasi) secara berdampingan, dengan mengonversinya ke RGB agar dapat ditampilkan dengan benar.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("Perbandingan 2 gambar pertama (Light Noise Reduced vs Channel-wise Normalized):")

for i in range(2):
    img_before_norm = dataset_cv_step5[i] # This is BGR uint8
    img_after_norm = dataset_cv_step6[i]  # This is RGB float with (img - mean) / std

    # Convert BGR to RGB for plotting img_before_norm
    img_before_norm_rgb = cv2.cvtColor(img_before_norm, cv2.COLOR_BGR2RGB)

    # For displaying the normalized image, clip values to [0, 1] as imshow expects this range for float images.
    # Values outside this range will be clipped by imshow automatically, but explicit clipping ensures consistency.
    img_after_norm_display = np.clip(img_after_norm, 0, 1)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_before_norm_rgb)
    plt.title(f"5. Light Noise Reduced {i+1}")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(img_after_norm_display)
    plt.title(f"6. Channel-wise Normalized {i+1}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

**Perbandingan sebelum preprocessing dan setelah preprocessing**

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

print("Perbandingan 2 gambar pertama (Image Asli vs Setelah Preprocessing):")

for i in range(2):
    img_original = dataset_cv_initial[i]
    img_preprocessed = dataset_cv_step6[i]
    img_original_rgb = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)
    img_preprocessed_display = np.clip(img_preprocessed, 0, 1)
    plt.figure(figsize=(12, 6))

    # Image Asli
    plt.subplot(1, 2, 1)
    plt.imshow(img_original_rgb)
    plt.title(f"Image Asli {i+1}")
    plt.axis('off')

    # Image Setelah Preprocessing
    plt.subplot(1, 2, 2)
    plt.imshow(img_preprocessed_display)
    plt.title(f"Setelah Preprocessing {i+(1)}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import cv2
import numpy as np

rows = []

for i in range(5):
    img_before = dataset_cv_step5[i]   # BGR uint8
    img_after  = dataset_cv_step6[i]   # RGB float normalized

    # Samakan ke RGB
    img_before_rgb = cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB)

    rows.append({
        "Gambar": i + 1,
        "Pixel_(0,0)_Before_RGB": img_before_rgb[0, 0].tolist(),
        "Pixel_(0,0)_After_RGB": img_after[0, 0].tolist()
    })

df_rgb = pd.DataFrame(rows)
df_rgb